In [1]:
import os
import json
import traceback
from datetime import datetime
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType, TimestampType
)
from pyspark.sql.window import Window

# ── Konfigurasi HDFS ──────────────────────────────────────────────────────────
HDFS_HOST      = "100.74.49.87"
HDFS_PORT      = 8020
HDFS_BASE      = f"hdfs://{HDFS_HOST}:{HDFS_PORT}"
HDFS_API_DIR   = f"{HDFS_BASE}/data/saham/api/"
HDFS_RSS_DIR   = f"{HDFS_BASE}/data/saham/rss/"
HDFS_HASIL_DIR = f"{HDFS_BASE}/data/saham/hasil/"

# ── Output lokal ──────────────────────────────────────────────────────────────
DASHBOARD_DIR  = Path("../dashboard/data")
DASHBOARD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON    = DASHBOARD_DIR / "spark_results.json"

# ── Perusahaan target untuk Analisis 3 ────────────────────────────────────────
TARGET_COMPANIES = {
    "BCA"    : ["BCA", "Bank Central Asia"],
    "BRI"    : ["BRI", "Bank Rakyat Indonesia"],
    "Telkom" : ["Telkom", "TLKM"],
    "Astra"  : ["Astra", "ASII"],
    "Mandiri": ["Mandiri", "Bank Mandiri"],
}

print("✅ Import selesai")
print(f"📁 Dashboard output : {OUTPUT_JSON.resolve()}")
print(f"🌐 HDFS Base        : {HDFS_BASE}")

✅ Import selesai
📁 Dashboard output : /mnt/c/Users/Oryza/SEM4/kelompok-3-ets-bigdata/dashboard/data/spark_results.json
🌐 HDFS Base        : hdfs://100.74.49.87:8020


In [2]:
spark = (
    SparkSession.builder
    .appName("SahamMeter-ETS-Kelompok3")
    # ── HDFS DataNode routing ──────────────────────────────────────────────
    .config("spark.hadoop.dfs.datanode.hostname",                "100.74.49.87")
    .config("spark.hadoop.dfs.client.use.datanode.hostname",     "true")
    .config("spark.hadoop.dfs.datanode.use.datanode.hostname",   "true")
    # ── Override IP internal Docker → IP Tailscale ─────────────────────────
    .config("spark.hadoop.fs.hdfs.impl",
            "org.apache.hadoop.hdfs.DistributedFileSystem")
    .config("spark.hadoop.fs.AbstractFileSystem.hdfs.impl",
            "org.apache.hadoop.fs.Hdfs")
    # ── Timeout lebih pendek (ms) ──────────────────────────────────────────
    .config("spark.hadoop.dfs.client.socket-timeout",            "10000")
    .config("spark.hadoop.ipc.client.connect.timeout",           "10000")
    .config("spark.network.timeout",                             "120s")
    # ── JSON & parsing ─────────────────────────────────────────────────────
    .config("spark.sql.legacy.timeParserPolicy",                 "LEGACY")
    .config("spark.sql.jsonGenerator.ignoreNullFields",          "false")
    # ── Misc ───────────────────────────────────────────────────────────────
    .config("spark.hadoop.fs.hdfs.impl.disable.cache",           "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print(f"✅ SparkSession aktif : {spark.version}")
print(f"   App Name          : {spark.sparkContext.appName}")
print(f"   Master            : {spark.sparkContext.master}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/04 15:21:14 WARN Utils: Your hostname, LAPTOP-E93TABFG, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/04 15:21:14 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/04 15:21:24 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


✅ SparkSession aktif : 4.1.1
   App Name          : SahamMeter-ETS-Kelompok3
   Master            : local[*]


In [3]:
def hdfs_list_files(hdfs_path: str) -> list[str]:
    """List semua file di direktori HDFS. Return list path string."""
    try:
        jvm    = spark._jvm
        conf   = spark._jsc.hadoopConfiguration()
        fs     = jvm.org.apache.hadoop.fs.FileSystem.get(
                     jvm.java.net.URI(hdfs_path), conf)
        status = fs.listStatus(jvm.org.apache.hadoop.fs.Path(hdfs_path))
        paths  = [s.getPath().toString() for s in status]
        return paths
    except Exception as e:
        print(f"⚠️  HDFS list gagal untuk {hdfs_path}: {e}")
        return []


def hdfs_path_exists(hdfs_path: str) -> bool:
    """Cek apakah path HDFS ada."""
    try:
        jvm  = spark._jvm
        conf = spark._jsc.hadoopConfiguration()
        fs   = jvm.org.apache.hadoop.fs.FileSystem.get(
                   jvm.java.net.URI(hdfs_path), conf)
        return fs.exists(jvm.org.apache.hadoop.fs.Path(hdfs_path))
    except Exception as e:
        print(f"⚠️  HDFS exists check gagal: {e}")
        return False


# ── Test koneksi ───────────────────────────────────────────────────────────────
print("🔍 Test koneksi HDFS...")
for label, path in [("API", HDFS_API_DIR), ("RSS", HDFS_RSS_DIR), ("Hasil", HDFS_HASIL_DIR)]:
    exists = hdfs_path_exists(path)
    files  = hdfs_list_files(path) if exists else []
    status = f"✅ Ada, {len(files)} file" if exists else "❌ Tidak ditemukan"
    print(f"   [{label:5s}] {path}  →  {status}")
    for f in files[:5]:
        print(f"            └─ {f.split('/')[-1]}")
    if len(files) > 5:
        print(f"            └─ ... dan {len(files)-5} file lainnya")

🔍 Test koneksi HDFS...
   [API  ] hdfs://100.74.49.87:8020/data/saham/api/  →  ✅ Ada, 40 file
            └─ 2026-05-04_14-25-37.json
            └─ 2026-05-04_14-30-41.json
            └─ 2026-05-04_14-35-41.json
            └─ 2026-05-04_14-40-41.json
            └─ 2026-05-04_14-41-10.json
            └─ ... dan 35 file lainnya
   [RSS  ] hdfs://100.74.49.87:8020/data/saham/rss/  →  ✅ Ada, 23 file
            └─ 2026-05-04_14-23-37.json
            └─ 2026-05-04_14-28-42.json
            └─ 2026-05-04_14-38-44.json
            └─ 2026-05-04_14-41-11.json
            └─ 2026-05-04_14-48-05.json
            └─ ... dan 18 file lainnya
   [Hasil] hdfs://100.74.49.87:8020/data/saham/hasil/  →  ✅ Ada, 9 file
            └─ frekuensi_20260504_143744
            └─ frekuensi_20260504_143927
            └─ frekuensi_20260504_145033
            └─ return_20260504_143744
            └─ return_20260504_143927
            └─ ... dan 4 file lainnya


In [4]:
schema_saham = StructType([
    StructField("ticker",       StringType(),  True),
    StructField("harga",        DoubleType(),  True),
    StructField("open",         DoubleType(),  True),
    StructField("high",         DoubleType(),  True),
    StructField("low",          DoubleType(),  True),
    StructField("volume",       LongType(),    True),
    StructField("change_pct",   DoubleType(),  True),
    StructField("is_simulated", StringType(),  True),
    StructField("timestamp",    StringType(),  True),
])

schema_berita = StructType([
    StructField("id",           StringType(), True),
    StructField("judul",        StringType(), True),
    StructField("ringkasan",    StringType(), True),
    StructField("sentimen",     StringType(), True),
    StructField("sumber",       StringType(), True),
    StructField("timestamp",    StringType(), True),
    StructField("url",          StringType(), True),
    StructField("waktu_terbit", StringType(), True),
])

print("✅ Schema didefinisikan")
spark.createDataFrame([], schema_saham).printSchema()
spark.createDataFrame([], schema_berita).printSchema()

✅ Schema didefinisikan
root
 |-- ticker: string (nullable = true)
 |-- harga: double (nullable = true)
 |-- open: double (nullable = true)
 |-- high: double (nullable = true)
 |-- low: double (nullable = true)
 |-- volume: long (nullable = true)
 |-- change_pct: double (nullable = true)
 |-- is_simulated: string (nullable = true)
 |-- timestamp: string (nullable = true)

root
 |-- id: string (nullable = true)
 |-- judul: string (nullable = true)
 |-- ringkasan: string (nullable = true)
 |-- sentimen: string (nullable = true)
 |-- sumber: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- url: string (nullable = true)
 |-- waktu_terbit: string (nullable = true)



In [5]:
print("📂 Membaca data saham dari HDFS...")
df_saham = None
SAHAM_DARI_HDFS = False

try:
    if not hdfs_path_exists(HDFS_API_DIR):
        raise FileNotFoundError(f"Path tidak ditemukan: {HDFS_API_DIR}")

    file_list  = hdfs_list_files(HDFS_API_DIR)
    json_files = [f for f in file_list if f.endswith(".json")]

    if not json_files:
        raise ValueError("Tidak ada file .json di direktori HDFS API")

    print(f"   Ditemukan {len(json_files)} file JSON")

    df_saham = (
        spark.read
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema_saham)
        .json(HDFS_API_DIR + "*.json")
    )

    count = df_saham.count()
    if count == 0:
        raise ValueError("File JSON ada tapi tidak ada baris data")

    SAHAM_DARI_HDFS = True
    print(f"   ✅ Berhasil baca {count} baris data saham dari HDFS")

except Exception as e:
    print(f"   ❌ Gagal baca dari HDFS: {e}")
    df_saham = buat_dummy_saham(spark)

# Sesuaikan dropna dengan nama field asli
df_saham = df_saham.dropna(subset=["ticker", "harga", "open"])

print(f"\n📊 Preview data saham ({'HDFS' if SAHAM_DARI_HDFS else 'DUMMY'}):")
df_saham.show(10, truncate=False)

📂 Membaca data saham dari HDFS...
   Ditemukan 40 file JSON


26/05/04 15:21:51 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://100.74.49.87:8020/data/saham/api/*.json.
java.io.FileNotFoundException: File does not exist: hdfs://100.74.49.87:8020/data/saham/api/*.json
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala

   ✅ Berhasil baca 974 baris data saham dari HDFS

📊 Preview data saham (HDFS):


[Stage 4:>                                                          (0 + 1) / 1]

+------+--------+--------+--------+--------+-------+----------+------------+--------------------------+
|ticker|harga   |open    |high    |low     |volume |change_pct|is_simulated|timestamp                 |
+------+--------+--------+--------+--------+-------+----------+------------+--------------------------+
|BBCA  |9465.68 |9568.08 |9593.14 |9415.2  |6070000|-1.0702   |true        |2026-04-30T01:13:00.882441|
|BBRI  |4822.45 |4774.41 |4832.67 |4739.59 |5908337|1.0062    |true        |2026-04-30T01:13:01.654754|
|TLKM  |3863.51 |3880.83 |3911.34 |3834.9  |4058963|-0.4463   |true        |2026-04-30T01:13:01.945977|
|ASII  |5155.32 |5200.31 |5231.47 |5134.96 |3966514|-0.8651   |true        |2026-04-30T01:13:02.035776|
|BMRI  |6128.38 |6048.25 |6143.28 |6037.97 |7601020|1.3248    |true        |2026-04-30T01:13:02.132570|
|UNVR  |2095.05 |2086.4  |2103.79 |2084.13 |2685177|0.4146    |true        |2026-04-30T01:13:02.276299|
|GOTO  |87.47   |88.54   |89.19   |87.39   |1212264|-1.2085   |t

In [6]:
# Baca tanpa schema dulu, biar Spark auto-detect
df_raw = spark.read.option("multiLine", "true").json(HDFS_API_DIR + "*.json")
df_raw.printSchema()
df_raw.show(3, truncate=False)


26/05/04 15:22:02 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://100.74.49.87:8020/data/saham/api/*.json.
java.io.FileNotFoundException: File does not exist: hdfs://100.74.49.87:8020/data/saham/api/*.json
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala

root
 |-- change_pct: double (nullable = true)
 |-- harga: double (nullable = true)
 |-- high: double (nullable = true)
 |-- is_simulated: boolean (nullable = true)
 |-- low: double (nullable = true)
 |-- open: double (nullable = true)
 |-- ticker: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- volume: long (nullable = true)

+----------+-------+-------+------------+-------+-------+------+--------------------------+-------+
|change_pct|harga  |high   |is_simulated|low    |open   |ticker|timestamp                 |volume |
+----------+-------+-------+------------+-------+-------+------+--------------------------+-------+
|-1.0702   |9465.68|9593.14|true        |9415.2 |9568.08|BBCA  |2026-04-30T01:13:00.882441|6070000|
|1.0062    |4822.45|4832.67|true        |4739.59|4774.41|BBRI  |2026-04-30T01:13:01.654754|5908337|
|-0.4463   |3863.51|3911.34|true        |3834.9 |3880.83|TLKM  |2026-04-30T01:13:01.945977|4058963|
+----------+-------+-------+------------+-------

In [7]:
df_agg = df_saham.groupBy("ticker").agg(
    F.first("open",  ignorenulls=True).alias("harga_awal"),
    F.last("harga",  ignorenulls=True).alias("harga_terkini"),
    F.max("high").alias("harga_tertinggi"),
    F.min("low").alias("harga_terendah"),
    F.sum("volume").alias("total_volume"),
)

df_return = df_agg.withColumn(
    "return_pct",
    F.round(
        (F.col("harga_terkini") - F.col("harga_awal")) / F.col("harga_awal") * 100,
        4
    )
).withColumn(
    "status",
    F.when(F.col("return_pct") > 0, "NAIK")
     .when(F.col("return_pct") < 0, "TURUN")
     .otherwise("FLAT")
).orderBy(F.col("return_pct").desc())

df_return.select(
    "ticker", "harga_awal", "harga_terkini", "return_pct", "status", "total_volume"
).show(truncate=False)

result_return = df_return.toPandas().to_dict(orient="records")
print(f"✅ Analisis 1 selesai — {len(result_return)} saham diproses")

+------+----------+-------------+----------+------+------------+
|ticker|harga_awal|harga_terkini|return_pct|status|total_volume|
+------+----------+-------------+----------+------+------------+
|PGAS  |1440.79   |1895.0       |31.5251   |NAIK  |3317620431  |
|ASII  |5200.31   |6100.0       |17.3007   |NAIK  |2730171930  |
|INDF  |7243.99   |6975.0       |-3.7133   |TURUN |1491133512  |
|UNVR  |2086.4    |1560.0       |-25.2301  |TURUN |1855584224  |
|TLKM  |3880.83   |2870.0       |-26.0467  |TURUN |7586764661  |
|BMRI  |6048.25   |4420.0       |-26.921   |TURUN |10504453160 |
|BBRI  |4774.41   |3030.0       |-36.5367  |TURUN |14960384468 |
|BBCA  |9568.08   |5900.0       |-38.3366  |TURUN |10470307136 |
|ICBP  |11386.39  |6775.0       |-40.4991  |TURUN |563273899   |
|GOTO  |88.54     |51.0         |-42.3989  |TURUN |232656787186|
+------+----------+-------------+----------+------+------------+

✅ Analisis 1 selesai — 10 saham diproses


In [8]:
df_volatilitas = df_saham.groupBy("ticker").agg(
    F.round(F.stddev("harga"), 4).alias("stddev_harga"),
    F.round(F.avg("harga"), 4).alias("rata_rata_harga"),
    F.max("high").alias("high"),
    F.min("low").alias("low"),
    F.count("*").alias("jumlah_snapshot"),
).withColumn(
    "range_intraday",
    F.round(F.col("high") - F.col("low"), 2)
).withColumn(
    "cv_pct",
    F.round(
        F.when(F.col("rata_rata_harga") > 0,
               F.col("stddev_harga") / F.col("rata_rata_harga") * 100)
         .otherwise(0.0),
        4
    )
).withColumn(
    "level_volatilitas",
    F.when(F.col("cv_pct") > 2.0, "TINGGI")
     .when(F.col("cv_pct") > 0.5, "SEDANG")
     .otherwise("RENDAH")
).orderBy(F.col("cv_pct").desc())

df_volatilitas.show(truncate=False)

result_volatilitas = df_volatilitas.toPandas().fillna(0).to_dict(orient="records")
print(f"✅ Analisis 2 selesai — {len(result_volatilitas)} saham diproses")

+------+------------+---------------+--------+-------+---------------+--------------+-------+-----------------+
|ticker|stddev_harga|rata_rata_harga|high    |low    |jumlah_snapshot|range_intraday|cv_pct |level_volatilitas|
+------+------------+---------------+--------+-------+---------------+--------------+-------+-----------------+
|GOTO  |17.573      |63.2453        |89.69   |50.0   |96             |39.69         |27.7855|TINGGI           |
|ICBP  |2218.5083   |8300.7885      |11834.69|6675.0 |95             |5159.69       |26.7265|TINGGI           |
|BBCA  |1718.9774   |7116.1903      |9902.5  |5800.0 |97             |4102.5        |24.1559|TINGGI           |
|BBRI  |855.0359    |3638.0862      |4924.91 |3000.0 |97             |1924.91       |23.5024|TINGGI           |
|BMRI  |779.2128    |4992.5891      |6204.06 |4410.0 |98             |1794.06       |15.6074|TINGGI           |
|UNVR  |263.3074    |1744.7197      |2189.62 |1550.0 |96             |639.62        |15.0917|TINGGI     

In [9]:
def buat_dummy_berita(spark):
    print("   ⚠️  Menggunakan data DUMMY untuk berita")
    dummy = [
        ("1", "BCA catat laba bersih Rp 48 triliun", "", "positif", "CNBC", "2025-04-01", "https://cnbc.id/1", "2025-04-01"),
        ("2", "BRI ekspansi kredit UMKM", "", "positif", "Bisnis", "2025-04-01", "https://bisnis.id/1", "2025-04-01"),
        ("3", "Telkom luncurkan layanan 5G", "", "positif", "Detik", "2025-04-01", "https://detik.id/1", "2025-04-01"),
        ("4", "Astra genjot penjualan EV", "", "netral", "Kontan", "2025-04-01", "https://kontan.id/1", "2025-04-01"),
        ("5", "Mandiri syariah berencana IPO", "", "positif", "Tempo", "2025-04-01", "https://tempo.id/1", "2025-04-01"),
    ]
    return spark.createDataFrame(dummy, schema=schema_berita)

print("📰 Membaca data berita dari HDFS...")
df_berita = None
BERITA_DARI_HDFS = False

try:
    if not hdfs_path_exists(HDFS_RSS_DIR):
        raise FileNotFoundError(f"Path tidak ditemukan: {HDFS_RSS_DIR}")

    file_list  = hdfs_list_files(HDFS_RSS_DIR)
    json_files = [f for f in file_list if f.endswith(".json")]

    if not json_files:
        raise ValueError("Tidak ada file .json di direktori HDFS RSS")

    print(f"   Ditemukan {len(json_files)} file JSON")

    df_berita = (
        spark.read
        .option("multiLine", "true")
        .option("mode", "PERMISSIVE")
        .schema(schema_berita)
        .json(HDFS_RSS_DIR + "*.json")
    )

    count = df_berita.count()
    if count == 0:
        raise ValueError("File JSON ada tapi tidak ada baris data")

    BERITA_DARI_HDFS = True
    print(f"   ✅ Berhasil baca {count} baris data berita dari HDFS")

except Exception as e:
    print(f"   ❌ Gagal baca dari HDFS: {e}")
    df_berita = buat_dummy_berita(spark)

df_berita = df_berita.dropna(subset=["judul"])

print(f"\n📰 Preview data berita ({'HDFS' if BERITA_DARI_HDFS else 'DUMMY'}):")
df_berita.select("judul", "sentimen", "sumber", "waktu_terbit").show(10, truncate=55)

📰 Membaca data berita dari HDFS...
   Ditemukan 23 file JSON


26/05/04 15:22:12 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://100.74.49.87:8020/data/saham/rss/*.json.
java.io.FileNotFoundException: File does not exist: hdfs://100.74.49.87:8020/data/saham/rss/*.json
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1842)
	at org.apache.hadoop.hdfs.DistributedFileSystem$29.doCall(DistributedFileSystem.java:1835)
	at org.apache.hadoop.fs.FileSystemLinkResolver.resolve(FileSystemLinkResolver.java:81)
	at org.apache.hadoop.hdfs.DistributedFileSystem.getFileStatus(DistributedFileSystem.java:1850)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala

   ✅ Berhasil baca 1844 baris data berita dari HDFS

📰 Preview data berita (HDFS):
+-------------------------------------------------------+--------+--------------------------------------+-------------------------------+
|                                                  judul|sentimen|                                sumber|                   waktu_terbit|
+-------------------------------------------------------+--------+--------------------------------------+-------------------------------+
|Tips Pilih Perlindungan Kendaraan di Era Mobil Konve...|  netral|CNN Indonesia | Berita Terkini Ekonomi|Thu, 30 Apr 2026 00:02:18 +0700|
|Bank Mega Salurkan Bantuan Sembako ke Korban Bencana...|  netral|CNN Indonesia | Berita Terkini Ekonomi|Wed, 29 Apr 2026 23:15:33 +0700|
|  Iran Resmi Larang Ekspor Baja Gegara Agresi AS-Israel|  netral|CNN Indonesia | Berita Terkini Ekonomi|Wed, 29 Apr 2026 22:00:55 +0700|
|TASPEN Beri Santunan ke Ahli Waris Guru Korban Kecel...|  netral|CNN Indonesia | Berita 

In [10]:
print("━" * 60)
print("📰 ANALISIS 3: Frekuensi Sebutan Perusahaan di Berita")
print("━" * 60)

df_tagged = df_berita

for company, keywords in TARGET_COMPANIES.items():
    pattern  = "|".join(keywords)
    col_name = f"mention_{company.lower()}"
    df_tagged = df_tagged.withColumn(
        col_name,
        F.when(
            F.regexp_extract(F.upper(F.col("judul")), pattern.upper(), 0) != "",
            1
        ).otherwise(0)
    )

print("\nSample tagging berita:")
df_tagged.select(
    "judul",
    *[f"mention_{c.lower()}" for c in TARGET_COMPANIES]
).show(10, truncate=50)

# Hitung total mention per perusahaan
mention_counts = {}
for company in TARGET_COMPANIES:
    col_name = f"mention_{company.lower()}"
    count = df_tagged.agg(F.sum(col_name).cast("long")).collect()[0][0] or 0
    mention_counts[company] = int(count)

# Buat DataFrame hasil
rows_frekuensi = [
    (company, int(count), list(TARGET_COMPANIES[company]))
    for company, count in sorted(mention_counts.items(), key=lambda x: -x[1])
]

df_frekuensi = spark.createDataFrame(
    [(r[0], r[1]) for r in rows_frekuensi],
    schema=StructType([
        StructField("perusahaan",     StringType(), True),
        StructField("jumlah_sebutan", LongType(),   True),
    ])
)

print("\n🏆 Peringkat sebutan perusahaan di berita:")
df_frekuensi.show()

print("\n📊 Distribusi sentimen keseluruhan berita:")
df_berita.groupBy("sentimen").count().orderBy(F.col("count").desc()).show()

result_frekuensi = [
    {"perusahaan": r[0], "jumlah_sebutan": r[1], "keywords": r[2]}
    for r in rows_frekuensi
]

print(f"✅ Analisis 3 selesai")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📰 ANALISIS 3: Frekuensi Sebutan Perusahaan di Berita
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Sample tagging berita:


+--------------------------------------------------+-----------+-----------+--------------+-------------+---------------+
|                                             judul|mention_bca|mention_bri|mention_telkom|mention_astra|mention_mandiri|
+--------------------------------------------------+-----------+-----------+--------------+-------------+---------------+
|Tips Pilih Perlindungan Kendaraan di Era Mobil ...|          0|          0|             0|            0|              0|
|Bank Mega Salurkan Bantuan Sembako ke Korban Be...|          0|          0|             0|            0|              0|
|Iran Resmi Larang Ekspor Baja Gegara Agresi AS-...|          0|          0|             0|            0|              0|
|TASPEN Beri Santunan ke Ahli Waris Guru Korban ...|          0|          0|             0|            0|              0|
|PAM Jaya Minta Rumah Tersambung Pipa PDAM Setop...|          0|          0|             0|            0|              0|
|                       

+----------+--------------+
|perusahaan|jumlah_sebutan|
+----------+--------------+
|   Mandiri|            12|
|       BRI|             8|
|       BCA|             0|
|    Telkom|             0|
|     Astra|             0|
+----------+--------------+


📊 Distribusi sentimen keseluruhan berita:
+--------+-----+
|sentimen|count|
+--------+-----+
|  netral| 1680|
| positif|  120|
| negatif|   44|
+--------+-----+

✅ Analisis 3 selesai


In [11]:
print("💾 Menyimpan hasil analisis ke dashboard...")

output_payload = {
    "metadata": {
        "project"       : "SahamMeter",
        "kelompok"      : 3,
        "generated_at"  : datetime.now().isoformat(),
        "spark_version" : spark.version,
        "saham_dari_hdfs" : SAHAM_DARI_HDFS,
        "berita_dari_hdfs": BERITA_DARI_HDFS,
        "hdfs_base"     : HDFS_BASE,
    },
    "analisis_1_return": result_return,
    "analisis_2_volatilitas": result_volatilitas,
    "analisis_3_frekuensi_berita": result_frekuensi,
}

# Simpan ke file lokal
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(output_payload, f, ensure_ascii=False, indent=2, default=str)

size_kb = OUTPUT_JSON.stat().st_size / 1024
print(f"   ✅ Tersimpan ke : {OUTPUT_JSON.resolve()}")
print(f"   📦 Ukuran file  : {size_kb:.2f} KB")
print(f"   🕐 Timestamp    : {output_payload['metadata']['generated_at']}")


💾 Menyimpan hasil analisis ke dashboard...
   ✅ Tersimpan ke : /mnt/c/Users/Oryza/SEM4/kelompok-3-ets-bigdata/dashboard/data/spark_results.json
   📦 Ukuran file  : 6.10 KB
   🕐 Timestamp    : 2026-05-04T15:22:28.724064


In [12]:
print("☁️  Menyimpan hasil ke HDFS...")

timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")

def simpan_ke_hdfs(df, hdfs_path: str, label: str):
    """Simpan DataFrame ke HDFS sebagai JSON. Return True jika sukses."""
    try:
        (
            df.coalesce(1)            # 1 file output
            .write
            .mode("overwrite")
            .option("encoding", "UTF-8")
            .json(hdfs_path)
        )
        print(f"   ✅ [{label}] tersimpan ke {hdfs_path}")
        return True
    except Exception as e:
        print(f"   ❌ [{label}] gagal simpan ke HDFS: {e}")
        return False


# Definisi output HDFS per analisis
hdfs_outputs = [
    (df_return,      f"{HDFS_HASIL_DIR}return_{timestamp_str}",      "Return Saham"),
    (df_volatilitas, f"{HDFS_HASIL_DIR}volatilitas_{timestamp_str}", "Volatilitas"),
    (df_frekuensi,   f"{HDFS_HASIL_DIR}frekuensi_{timestamp_str}",   "Frekuensi Berita"),
]

hdfs_sukses = 0
for df, path, label in hdfs_outputs:
    ok = simpan_ke_hdfs(df, path, label)
    if ok:
        hdfs_sukses += 1

print(f"\n   Total tersimpan ke HDFS: {hdfs_sukses}/{len(hdfs_outputs)}")

if hdfs_sukses == 0:
    print("   ⚠️  Semua gagal ke HDFS — hasil tetap tersimpan di dashboard lokal")

☁️  Menyimpan hasil ke HDFS...


   ✅ [Return Saham] tersimpan ke hdfs://100.74.49.87:8020/data/saham/hasil/return_20260504_152228
   ✅ [Volatilitas] tersimpan ke hdfs://100.74.49.87:8020/data/saham/hasil/volatilitas_20260504_152228
   ✅ [Frekuensi Berita] tersimpan ke hdfs://100.74.49.87:8020/data/saham/hasil/frekuensi_20260504_152228

   Total tersimpan ke HDFS: 3/3


In [13]:
print("═" * 60)
print("  RINGKASAN HASIL — SahamMeter ETS Big Data Kelompok 3")
print("═" * 60)

print(f"\n📌 Sumber Data:")
print(f"   Saham  : {'HDFS ✅' if SAHAM_DARI_HDFS else 'DUMMY ⚠️'}")
print(f"   Berita : {'HDFS ✅' if BERITA_DARI_HDFS else 'DUMMY ⚠️'}")

print(f"\n📊 Analisis 1 — Return per Saham:")
for r in result_return:
    arrow = "▲" if r["return_pct"] > 0 else ("▼" if r["return_pct"] < 0 else "─")
    print(f"   {r['ticker']:6s} {arrow} {r['return_pct']:+.2f}%  "
          f"(Rp{r['harga_awal']:,.0f} → Rp{r['harga_terkini']:,.0f})")

print(f"\n📊 Analisis 2 — Volatilitas Intraday:")
for r in result_volatilitas:
    print(f"   {r['ticker']:6s}  stddev={r['stddev_harga']:.2f}  "
          f"range={r['range_intraday']:.0f}  [{r['level_volatilitas']}]")

print(f"\n📊 Analisis 3 — Frekuensi Sebutan Berita:")
for r in result_frekuensi:
    bar = "█" * r["jumlah_sebutan"]
    bar = bar if bar else "─"
    print(f"   {r['perusahaan']:8s} {bar} {r['jumlah_sebutan']}×")

print(f"\n💾 Output:")
print(f"   Lokal  : {OUTPUT_JSON.resolve()}")
print(f"   HDFS   : {HDFS_HASIL_DIR}")
print(f"\n✅ Semua analisis selesai — {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("═" * 60)

════════════════════════════════════════════════════════════
  RINGKASAN HASIL — SahamMeter ETS Big Data Kelompok 3
════════════════════════════════════════════════════════════

📌 Sumber Data:
   Saham  : HDFS ✅
   Berita : HDFS ✅

📊 Analisis 1 — Return per Saham:
   PGAS   ▲ +31.53%  (Rp1,441 → Rp1,895)
   ASII   ▲ +17.30%  (Rp5,200 → Rp6,100)
   INDF   ▼ -3.71%  (Rp7,244 → Rp6,975)
   UNVR   ▼ -25.23%  (Rp2,086 → Rp1,560)
   TLKM   ▼ -26.05%  (Rp3,881 → Rp2,870)
   BMRI   ▼ -26.92%  (Rp6,048 → Rp4,420)
   BBRI   ▼ -36.54%  (Rp4,774 → Rp3,030)
   BBCA   ▼ -38.34%  (Rp9,568 → Rp5,900)
   ICBP   ▼ -40.50%  (Rp11,386 → Rp6,775)
   GOTO   ▼ -42.40%  (Rp89 → Rp51)

📊 Analisis 2 — Volatilitas Intraday:
   GOTO    stddev=17.57  range=40  [TINGGI]
   ICBP    stddev=2218.51  range=5160  [TINGGI]
   BBCA    stddev=1718.98  range=4102  [TINGGI]
   BBRI    stddev=855.04  range=1925  [TINGGI]
   BMRI    stddev=779.21  range=1794  [TINGGI]
   UNVR    stddev=263.31  range=640  [TINGGI]
   TLKM    st